# This notebook compares eukaryotic taxa seen in shotgun metagenomes with those in metabarcoding sequence data. It is designed to use three different annotation outputs: 1) sourmash (read-based annotation of shotgun metagenomes), 2) taxonomic annotation (mmseqs-lca) and quantification (TPM) of assembled contigs, and 3) DADA2, BLAST, and MEGAN LCA for metabarcoding data. Sourmash databases were GTDB and MMETSP, while mmseqs and BLAST databases were NCBI-nr.

### This is currently written to assess taxa at the class level, but can be modified for other levels of taxonomic resolution.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import os as os
import glob as glob
import numpy as np
import functools as ft

## 01. Shotgun  metagenome sourmash results: Compile lists of observed taxa and calculate prokaryotic vs eukaryotic fractions

### Function to generate domain- and species-level data

In [3]:
def compile_taxa_lists(sourmash, dataset, stats):
    tax = pd.read_csv(sourmash)
    # Get total number of bps in metagenome data set
    bps_grouped = stats[['Dataset', '# bps (QCd)']].groupby(['Dataset']).sum()
    total_bps = bps_grouped.loc[dataset][0].astype(int)
    # Get fraction of metagenome for each domain (or "superkingdom")
    superk = tax[tax['rank'] == 'superkingdom']
    # Prokaryotes
    arch = superk[superk['lineage'].isin(['d__Archaea'])]['f_weighted_at_rank'].mean()
    bac = superk[superk['lineage'].isin(['d__Bacteria'])]['f_weighted_at_rank'].mean()
    prok_frac = arch + bac
    unass_frac = superk[superk['lineage'] == 'unclassified']['f_weighted_at_rank'].mean()
    if superk['lineage'].eq('Eukaryota').any():
        euk_frac = superk[superk['lineage'] == 'Eukaryota']['f_weighted_at_rank'].mean()
    else:
        euk_frac = 0
    # Calculate # bps per domain in metagenome
    prok_bps = prok_frac * total_bps
    prok_bps = prok_bps.round(0).astype(int)
    unass_bps = unass_frac * total_bps
    unass_bps = unass_bps.round(0).astype(int)
    euk_bps = euk_frac * total_bps
    euk_bps = euk_bps.round(0).astype(int)
    # Make dataframe with sample name and bps
    dataset_bps = pd.DataFrame({'Dataset' : pd.Series(dataset, dtype='str'), 
                               'total_bps': pd.Series(total_bps, dtype='int'),
                               'prok_bps': pd.Series(prok_bps, dtype='int'),
                               'euk_bps': pd.Series(euk_bps, dtype='int'),
                               'unass_bps': pd.Series(unass_bps, dtype='int')}) 
    tax['Dataset'] = dataset
    tax = tax[['Dataset', 'lineage', 'rank', 'f_weighted_at_rank']].groupby(['Dataset', 'lineage', 'rank']).sum().reset_index()
    tax = tax[tax['rank'] == 'species'] # this can be changed to any taxonomic level
    lineage = tax[['lineage', 'f_weighted_at_rank']].copy()
    # n should be changed to however many ranks are in the desired taxonomic level
    lineage[['Domain', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']] = lineage['lineage'].str.split(';', 
                                                                n=7, expand=True) 
    # Filter for eukaryotic classifications only
    euks = lineage.loc[lineage['Domain'] == 'Eukaryota']
    # Get taxa with unknown phyla
    unknown = euks[euks['Phylum'] == 'Unknown']
    # Change this to whatever taxonomic rank; currently set at 'Phylum'
    tax_level = euks[['Phylum','Class','Order', 'Family', 'Genus', 'Species', 'f_weighted_at_rank']].set_index('Phylum')
    # Multiply weighted fraction of taxon by total bps to get metagenome fraction of taxon
    tax_level['bp_fraction'] = tax_level['f_weighted_at_rank'] * euk_bps
    tax_level = tax_level.drop(columns=['f_weighted_at_rank'])
    tax_level['Metagenome Dataset'] = dataset
    return(tax_level, dataset_bps, unknown)

### Apply functions to all samples in all directories

In [5]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/sourmash/summaries/dataset")

In [6]:
stats = pd.read_csv("mgn_seq_stats.csv")
bps_grouped = stats[['Dataset', '# bps (QCd)']].groupby(['Dataset']).sum()
bps_grouped

,# bps (QCd)
Dataset,
CANON2018,5.814207e+12
RREAS2018,6.169170e+11
RREAS2019,3.098577e+11


In [7]:
# empty list for taxon-level observations
df_tax_mgns = []
# empty list for all eukaryotic reads and base pairs
df_sample_bps = []
# unknown
unknown_all = []

for file in glob.glob('*summarized.csv'):
    data, b = file.split('.', maxsplit=1)
    tax, bps, unknown = compile_taxa_lists(file, data, stats)
    df_tax_mgns.append(tax)
    df_sample_bps.append(bps)
    unknown_all.append(unknown)

df_tax_mgns = pd.concat(df_tax_mgns).reset_index()
df_mgns_bps = pd.concat(df_sample_bps)
df_mgns_bps['euk_bp_fraction'] = df_mgns_bps['euk_bps'] / df_mgns_bps['total_bps']
df_mgns_bps['prok_bp_fraction'] = df_mgns_bps['prok_bps'] / df_mgns_bps['total_bps']
df_mgns_bps['unass_bp_fraction'] = df_mgns_bps['unass_bps'] / df_mgns_bps['total_bps']
unknown_all = pd.concat(unknown_all)

#### Calculate average fraction of eukaryotic base pairs across all samples

In [8]:
df_mgns_bps.head()

,Dataset,total_bps,prok_bps,euk_bps,unass_bps,euk_bp_fraction,prok_bp_fraction,unass_bp_fraction
0,RREAS2018,616916961237,144911899794,11198978918,460806082525,0.018153,0.234897,0.746950
0,CANON2018,5814206629362,1533773518147,35948696461,4244484414753,0.006183,0.263798,0.730020
0,RREAS2019,309857691517,57016114617,10504129507,242868476378,0.033900,0.184007,0.783807


In [9]:
avg_euk_percent = np.average(df_mgns_bps['euk_bp_fraction'])
avg_prok_percent = np.average(df_mgns_bps['prok_bp_fraction'])
avg_unass_percent = np.average(df_mgns_bps['unass_bp_fraction'])
avg_euk_bps = np.average(df_mgns_bps['euk_bps'])
avg_prok_bps = np.average(df_mgns_bps['prok_bps'])
avg_unass_bps = np.average(df_mgns_bps['unass_bps'])
avg_prok_bps

np.float64(578567177519.3334)

#### Calculate average fraction of total reads per sample from fraction of base pairs

In [10]:
mgn_stats = pd.read_csv("mgn_seq_stats.csv")
avg_bps = np.average(mgn_stats['# bps (QCd)'])
avg_bps

np.float64(30921932486.77339)

#### Create a data frame with the average number of reads, average fraction of eukaryotic base pairs, and average fraction of eukaryotic reads for metagenomes

In [11]:
euk_frac_mgns = pd.DataFrame({'Sequence type': 'Metagenome reads', 
                              'Avg num bps or contigs': [avg_bps],
                           'Avg % euk bps or contigs' : [avg_euk_percent], 
                              'Avg % prok bps or contigs' : [avg_prok_percent],
                             'Avg % unass bps or contigs' : [avg_unass_percent],
                              'Avg num euk bps or contigs' : [avg_euk_bps],
                              'Avg num prok bps or contigs' : [avg_prok_bps],
                              'Avg num unass bps or contigs' : [avg_unass_bps]})

In [12]:
euk_frac_mgns

,Sequence type,Avg num bps or contigs,Avg % euk bps or contigs,Avg % prok bps or contigs,Avg % unass bps or contigs,Avg num euk bps or contigs,Avg num prok bps or contigs,Avg num unass bps or contigs
0,Metagenome reads,3.092193e+10,0.019412,0.227567,0.753592,1.921727e+10,5.785672e+11,1.649386e+12


In [13]:
euk_frac_mgns.to_csv("Metagenome_eukaryotic_reads_bps.csv", index=None)

## 02. Shotgun  metagenome contig annotation and associated TPMs: Compile fractions of observed taxa and calculate prokaryotic vs eukaryotic fractions

In [5]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/mmseqs/coverm-lca-filt-ALL") 

## Combine metagenome coverage by contig with taxonomic annotation by contig

### Function to identify matching LCA taxonomy and coverage files

In [6]:
def find_matching_file_by_prefix(filename, search_dir, old_suffix='-mmseqs_uniref_lca_filt.tsv', new_suffix='_covermcontig-out'):
    base = os.path.basename(filename)
    
    if base.endswith(old_suffix):
        # Strip the old suffix to get the prefix
        prefix = base[:-len(old_suffix)]
        # Construct the new filename
        match_name = prefix + new_suffix
        match_path = os.path.join(search_dir, match_name)
        if os.path.exists(match_path):
            return match_name
        else:
            return None
    else:
        return None

### Check all files have matches

In [7]:
for file in glob.glob('*filt.tsv'):
    tpms = find_matching_file_by_prefix(file, './')
    print(file, tpms)

### Function to process mmseqs output files only to check taxonomy

In [5]:
def check_mmseqs_taxonomy(lca):
    # Import LCA output
    tax_raw = pd.read_csv(lca, sep='\t')
    # Add sample name and drop irrelevant 'Kingdom' rank
    tax_raw['Sample'] = name
    tax_raw = tax_raw.drop('k', axis=1)

    # Remove trailing whitespace from all cells
    tax = tax_raw.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # set Domain to Unclassified if contig with coverage had no annotation
    tax['d'] = tax['d'].fillna(value='Unclassified')

    return(tax)

In [37]:
mmseqs_tax = []

for file in glob.glob('*filt.tsv'):
    name, b = file.rsplit('-', maxsplit=1)
    tax_comb = check_mmseqs_taxonomy(file)
    
    mmseqs_tax.append(tax_comb)

mmseqs_tax = pd.concat(mmseqs_tax).rename(columns={'Sample': 'Metagenome sample'})
mmseqs_tax.head()

,Contig,Tax_level,Best_tax,d,p,c,o,f,g,s,Metagenome sample
0,D0174A_S47_1874850,superkingdom,Eukaryota,Eukaryota,NaN,NaN,NaN,NaN,NaN,NaN,D0174A
1,D0174A_S47_1874851,no rank,unclassified,Unclassified,NaN,NaN,NaN,NaN,NaN,NaN,D0174A
2,D0174A_S47_1874854,no rank,unclassified,Unclassified,NaN,NaN,NaN,NaN,NaN,NaN,D0174A
3,D0174A_S47_1874855,no rank,unclassified,Unclassified,NaN,NaN,NaN,NaN,NaN,NaN,D0174A
4,D0174A_S47_1874856,no rank,unclassified,Unclassified,NaN,NaN,NaN,NaN,NaN,NaN,D0174A


In [38]:
df = mmseqs_tax.fillna('Unclassified')
df.head()

,Contig,Tax_level,Best_tax,d,p,c,o,f,g,s,Metagenome sample
0,D0174A_S47_1874850,superkingdom,Eukaryota,Eukaryota,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,D0174A
1,D0174A_S47_1874851,no rank,unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,D0174A
2,D0174A_S47_1874854,no rank,unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,D0174A
3,D0174A_S47_1874855,no rank,unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,D0174A
4,D0174A_S47_1874856,no rank,unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,D0174A


### Function to merge mmseqs taxonomy with contig coverages (TPM)

In [8]:
def merge_taxonomy_coverage(tpms, lca):
    # Contig coverage
    coverage = pd.read_csv(tpms, sep='\t', header=0, names=['Contig', 'TPM'])
    # Add column for sample name and remove contigs with TPM of 0
    name, b = tpms.split('_c', maxsplit=1)
    coverage['Sample'] = name
    coverage = coverage[coverage['TPM'] != 0.0]
    # Calculate total TPM value
    tpm_total = coverage['TPM'].sum()

    # Import LCA output
    tax_raw = pd.read_csv(lca, sep='\t')
    # Add sample name and drop irrelevant 'Kingdom' rank
    tax_raw['Sample'] = name
    tax = tax_raw.drop('k', axis=1)

    # Merge on contigs
    df = pd.merge(coverage, tax, on=['Contig', 'Sample'], how='outer')
    df = df[~df['TPM'].isna()]
    
    # Calculate percent TPM
    if tpm_total == 0:
        df['percent_TPM'] = 0
    else:
        df['percent_TPM'] = df['TPM'] / tpm_total

    # Drop raw TPM values
    df = df.drop(['TPM', 'Best_tax', 'Tax_level'], axis=1)
    # Need to get rid of NaNs for groupby
    df = df.fillna('Unclassified')
    
    # Make eukaryotic, prokaryotic, and unclassified domain ASV tables
    df_euks = df[df['d'] == 'Eukaryota'].drop(columns=['d'])
    df_proks = df[(df['d'] == 'Bacteria') | (df['d'] == 'Archaea')].drop(columns=['d'])
    df_unass = df[(df['d'] == 'Unclassified')]
    
    return(df, df_euks, df_proks, df_unass)

### Function to calculate total metagenome coverage

In [6]:
def get_tpm_sum(tpms):
    # Contig coverage
    coverage = pd.read_csv(tpms, sep='\t', header=0, names=['Contig', 'TPM'])
    # Add column for sample name and remove contigs with TPM of 0
    name, b = tpms.split('_c', maxsplit=1)
    coverage['Sample'] = name
    coverage = coverage[coverage['TPM'] != 0.0]
    # Calculate total TPM value
    tpm_total = coverage['TPM'].sum()
    print(name, tpm_total)

#### Check coverages of all metagenomes

In [31]:
for file in glob.glob('*out'):
    get_tpm_sum(file)

### Compile results from samples grouped by dataset

In [9]:
mmseqs_all = []
mmseqs_euk = []
mmseqs_prok = []
mmseqs_unass = []

for file in glob.glob('*filt.tsv'):
    name, b = file.rsplit('-', maxsplit=1)
    tpms = find_matching_file_by_prefix(file, './')
    all_tax, euks, proks, unass = merge_taxonomy_coverage(tpms, file)
    mmseqs_all.append(all_tax)
    mmseqs_euk.append(euks)
    mmseqs_prok.append(proks)
    mmseqs_unass.append(unass)

mmseqs_all = pd.concat(mmseqs_all).rename(columns={'Sample': 'Metagenome sample'})
mmseqs_euk = pd.concat(mmseqs_euk).rename(columns={'Sample': 'Metagenome sample'})
mmseqs_prok = pd.concat(mmseqs_prok).rename(columns={'Sample': 'Metagenome sample'})
mmseqs_unass = pd.concat(mmseqs_unass).rename(columns={'Sample': 'Metagenome sample'})

### Group by all taxonomic ranks to get overall TPM across samples

In [10]:
mmseqs_euk.head()

,Contig,Metagenome sample,p,c,o,f,g,s,percent_TPM
119,D0174A_S47_0120,D0174A,Bacillariophyta,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,0.000002
134,D0174A_S47_0135,D0174A,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,0.000003
183,D0174A_S47_0184,D0174A,Bacillariophyta,Coscinodiscophyceae,Thalassiosirales,Unclassified,Unclassified,Unclassified,0.000002
248,D0174A_S47_0249,D0174A,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,0.000002
265,D0174A_S47_0266,D0174A,Bacillariophyta,Unclassified,Unclassified,Unclassified,Unclassified,Unclassified,0.000001


In [11]:
mmseqs_euk[(mmseqs_euk['o'] == 'Prymnesiales') & (mmseqs_euk['c'] == 'Unclassified')]

,Contig,Metagenome sample,p,c,o,f,g,s,percent_TPM


### Group euks by sample at the phylum and class level for pairwise sample comparisons

In [12]:
mmseqs_euk_samps = mmseqs_euk.drop(['Contig', 'o', 'f', 'g', 's'], axis=1).groupby(['Metagenome sample', 'p', 'c']).sum().reset_index()
mmseqs_euk_samps

,Metagenome sample,p,c,percent_TPM
0,1903_buk5-1,Annelida,Clitellata,8.267616e-06
1,1903_buk5-1,Annelida,Polychaeta,2.758977e-05
2,1903_buk5-1,Annelida,Unclassified,3.004027e-06
3,1903_buk5-1,Apicomplexa,Aconoidasida,1.106824e-05
4,1903_buk5-1,Apicomplexa,Conoidasida,3.132316e-05
...,...,...,...,...
38518,zr2760_79,Zoopagomycota,Dimargaritomycetes,2.738130e-06
38519,zr2760_79,Zoopagomycota,Harpellomycetes,2.966331e-06
38520,zr2760_79,Zoopagomycota,Kickxellomycetes,3.006005e-06
38521,zr2760_79,Zoopagomycota,Unclassified,7.524395e-07


In [13]:
mmseqs_euk_samps.to_csv("contigs_mmseqs_Phylum_Class.csv", index=None)

### Aggregate taxonomy across all samples and calculate averages for Krona and stacked bars

In [14]:
mmseqs_all_drop = mmseqs_all.drop(['Contig', 'Metagenome sample'], axis=1)

# Sum across all samples first
mmseqs_all_grouped = mmseqs_all_drop.groupby(['d', 'p', 'c', 'o', 'f', 'g', 's']).sum().reset_index()
mmseqs_all_grouped.head()

,d,p,c,o,f,g,s,percent_TPM
0,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon,1.007062e-03
1,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon CG,9.880373e-07
2,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon CG01,2.632251e-06
3,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon ex4484,1.251314e-05
4,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmatarchaeota archaeon,4.131832e-06


#### Divide summed TPM values by the total number of samples for average TPMs

In [15]:
mmseqs_all_grouped['percent_TPM_corrected'] = mmseqs_all_grouped['percent_TPM'] / 218
mmseqs_all_grouped.head()

,d,p,c,o,f,g,s,percent_TPM,percent_TPM_corrected
0,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon,1.007062e-03,4.619551e-06
1,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon CG,9.880373e-07,4.532281e-09
2,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon CG01,2.632251e-06,1.207454e-08
3,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmarchaeota archaeon ex4484,1.251314e-05,5.739974e-08
4,Archaea,Candidatus Aenigmatarchaeota,Unclassified,Unclassified,Unclassified,Unclassified,Candidatus Aenigmatarchaeota archaeon,4.131832e-06,1.895336e-08


In [20]:
mmseqs_all_grouped['percent_TPM_corrected'].sum()

np.float64(1.0)

In [21]:
mmseqs_all_grouped[mmseqs_all_grouped['o'] == 'Prymnesiales']

,d,p,c,o,f,g,s,percent_TPM,percent_TPM_corrected
50327,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Chrysochromulinaceae,Chrysochromulina,Chrysochromulina acantha,2.122002e-05,9.733956e-08
50328,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Chrysochromulinaceae,Chrysochromulina,Chrysochromulina alifera,4.117341e-06,1.888688e-08
50329,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Chrysochromulinaceae,Chrysochromulina,Chrysochromulina cymbium,2.632073e-05,1.207373e-07
50330,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Chrysochromulinaceae,Chrysochromulina,Chrysochromulina parva,3.712407e-07,1.702939e-09
50331,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Chrysochromulinaceae,Chrysochromulina,Chrysochromulina sp. NIES-1333,2.198348e-04,1.008416e-06
50332,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Chrysochromulinaceae,Chrysochromulina,Chrysochromulina tobinii,2.699979e-01,1.238523e-03
50333,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Chrysochromulinaceae,Chrysochromulina,Unclassified,6.403544e-03,2.937405e-05
50334,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Prymnesiaceae,Haptolina,Haptolina brevifila,1.701339e-01,7.804306e-04
50335,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Prymnesiaceae,Haptolina,Haptolina ericina,1.046638e-01,4.801091e-04
50336,Eukaryota,Haptophyta,Prymnesiophyceae,Prymnesiales,Prymnesiaceae,Haptolina,Haptolina hirta,1.754896e-05,8.049981e-08


### Exported file has all tax ranks and can be reformatted for Krona in a different notebook

In [16]:
mmseqs_all_grouped.to_csv("CCE_2018-2019_contigs_mmseqs_TPM.csv", index=None)

### Get total contig count per sample

In [189]:
mmseqs_contigs = mmseqs_all[['Metagenome sample', 'Contig']]
mmseqs_contig_count = mmseqs_contigs.groupby('Metagenome sample').count()
mmseqs_contig_count.head()

,Contig
Metagenome sample,
1903_buk5-1,722829
1903_buk5-2,721343
1903_buk6-1,570377
1903_buk6-2,352602
1903buk1_0m-1,3246


In [208]:
mmseqs_contig_count.to_csv("CCE_2018-2019_mmseqs_total_contigs.csv", index=None)

### Calculate % contigs classified as prokaryote vs eukaryote vs unassigned

In [67]:
contig_euk_frac = mmseqs_euk[['Metagenome sample', 'percent_TPM']].rename(columns={'percent_TPM': 'euk_contig_percent'})
contig_euk_frac = contig_euk_frac.groupby('Metagenome sample').sum()
contig_prok_frac = mmseqs_prok[['Metagenome sample', 'percent_TPM']].rename(columns={'percent_TPM': 'prok_contig_percent'})
contig_prok_frac = contig_prok_frac.groupby('Metagenome sample').sum()
contig_unass_frac = mmseqs_unass[['Metagenome sample', 'percent_TPM']].rename(columns={'percent_TPM': 'unass_contig_percent'})
contig_unass_frac = contig_unass_frac.groupby('Metagenome sample').sum()
contig_prok_euk_frac_df = contig_euk_frac.merge(contig_prok_frac, left_index=True, right_index=True)
contig_frac_df = contig_prok_euk_frac_df.merge(contig_unass_frac, left_index=True, right_index=True)
contig_frac_df = contig_frac_df.merge(mmseqs_contig_count, left_index=True, right_index=True).rename(columns={'Contig': 'Total_contigs'})
contig_frac_df.head(5)

,euk_contig_percent,prok_contig_percent,unass_contig_percent,Total_contigs
Metagenome sample,,,,
1903_buk5-1,0.185821,0.304237,0.509942,722829
1903_buk5-2,0.177641,0.316295,0.506064,721343
1903_buk6-1,0.057686,0.554939,0.387375,570377
1903_buk6-2,0.050693,0.579238,0.370069,352602
1903buk1_0m-1,0.129014,0.494236,0.376750,3246


In [68]:
contig_frac_df.to_csv("mmseqs_contig_counts_by_domain_withunass.csv")

## 03. Metabarcoding data: Compile lists of observed taxa

### Function to calculate total number of reads and bps per sample, filter out bacterial and unclassified ASVs, and convert read counts to relative abundance values ('frequencies')

In [8]:
def counts_to_freq(file, marker_gene, read_len):
    asvs = pd.read_csv(file, header=0)
    # Move the taxonomy columns to the first positions in the df
    cols_to_move = ['Domain', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']
    asvs = asvs[ cols_to_move + [ col for col in asvs.columns if col not in cols_to_move ] ]
    # Convert non-sample columns to strings so sum function works
    asvs = asvs.astype({col: str for col in asvs.columns[0:9]})
    # Get total number of reads and bps per sample
    asvs.loc['total']= asvs.sum()
    sample_reads = asvs.iloc[-1:,9:].reset_index()
    sample_reads = pd.melt(sample_reads, id_vars='index')
    sample_reads = sample_reads.iloc[:,-2:].rename(columns=
                                       {'variable':'%s sample' % marker_gene, 
                                        'value':'total %s reads' % marker_gene})
    sample_reads['total %s bps' % marker_gene] = sample_reads['total %s reads' % marker_gene] * read_len # 180 bps for 18S, 313 bps for COI
   
    # Remove ASVs with no read counts in any samples
    asvs.loc[:, 'sum'] = asvs.iloc[:,9:].sum(axis=1)
    asvs_filt = asvs[asvs['sum'] != 0]
    
    # Replace 'nan' annotations with 'unassigned'
    asvs_filt = asvs_filt.replace('nan', 'unassigned')

    # Designate dinoflagellate class as Phylum Dinophyceae
    asvs_filt.loc[asvs_filt['Class'] == 'Dinophyceae', 'Phylum'] = 'Dinophyceae'
    
    # Correct class-level taxonomy for haptophytes, chlorophytes, and Oomycota
    asvs_filt.loc[asvs_filt['Order'] == 'Prymnesiales', 'Class'] = 'Prymnesiophyceae'
    asvs_filt.loc[asvs_filt['Order'] == 'Isochrysidales', 'Class'] = 'Prymnesiophyceae'
    asvs_filt.loc[asvs_filt['Order'] == 'Phaeocystales', 'Class'] = 'Prymnesiophyceae'
    asvs_filt.loc[asvs_filt['Order'] == 'Peronosporales', 'Class'] = 'Peronosporales'
    asvs_filt.loc[asvs_filt['Order'] == 'Saprolegniales', 'Class'] = 'Saprolegniales'
    asvs_filt.loc[asvs_filt['Order'] == 'Lagenidiales', 'Class'] = 'Lagenidiales'
    asvs_filt.loc[asvs_filt['Order'] == 'Pythiales', 'Class'] = 'Pythiales'
    asvs_filt.loc[asvs_filt['Family'] == 'Pycnococcaceae', 'Class'] = 'Pseudoscourfieldiales'
    asvs_filt.loc[asvs_filt['Order'] == 'Pseudoscourfieldiales', 'Class'] = 'Pyramimonadophyceae'
    asvs_filt.loc[asvs_filt['Order'] == 'Pyramimonadales', 'Class'] = 'Pyramimonadophyceae'
    
    # Calculate frequencies
    asvs_freq = asvs_filt.copy()
    asvs_freq.iloc[:,9:] = asvs_freq.iloc[:,9:] / asvs_freq.iloc[-1,9:]
    # Remove the column and row with ASV read count sums
    asvs_filt = asvs_filt.iloc[:-1,:-1]
    asvs_freq = asvs_freq.iloc[:-1,:-1]
    # Remove unnassigned ASV ID
    asvs_filt = asvs_filt[asvs_filt['ASV ID'] != 'unassigned']
    asvs_freq = asvs_freq[asvs_freq['ASV ID'] != 'unassigned']
    # Combine ASV IDs with all 'unassigned' values 
    asvs_filt = asvs_filt.groupby(['Domain', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']).sum().reset_index() 
    asvs_filt.loc[asvs_filt['ASV ID'].str.contains(r'(.*[0-9]ASV)'), 'ASV ID'] = 'ASVs_X'
    asvs_freq = asvs_freq.groupby(['Domain', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']).sum().reset_index() 
    asvs_freq.loc[asvs_freq['ASV ID'].str.contains(r'(.*[0-9]ASV)'), 'ASV ID'] = 'ASVs_X'
    return(asvs_filt, asvs_freq, sample_reads)

#### Excluding taxa with unknown class/genus filters out A LOT of ASVs. Currently I have opted to keep in unknown taxa.

### Function to reformat taxa lists

In [9]:
def reformat_ASV_table(df, marker_gene):
    samples = [col for col in df.columns if '1' in col] # Only sample column names have numbers
    cols_to_drop = ['Kingdom', 'ASV ID']
    df = df.drop(cols_to_drop, axis=1)
    taxon = [x for x in df.columns if '1' not in x]
    df_melt = pd.melt(df, id_vars=taxon, value_vars=samples)
    df_asvs = df_melt.rename(columns = {'variable':'%s sample' % marker_gene, 
                                        'value':'%s reads' % marker_gene})
    df_asvs = df_asvs[ (df_asvs['%s reads' % marker_gene] != 0.0) & (df_asvs['%s reads' % marker_gene].notna()) ]
    return(df_asvs)

### Test on one file

In [10]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/COI comparison/ASV tables")

In [11]:
file = 'RREAS2019_COI_asv_filtered_noCs_mgnmatches_tax.csv'
gene = 'COI'
len = 313

In [12]:
taxa_filt, taxa_freq, reads = counts_to_freq(file, gene, len)
#counts = reformat_ASV_table(taxa_filt, 'COI')

In [13]:
taxa_filt[taxa_filt['Class'] == 'Dinophyceae']

,Domain,Kingdom,Phylum,Class,Order,Family,Genus,Species,ASV ID,CN19_Nisk_SM_Night_5m_1,...,CN19_Nisk_Dav_Dawn_27m_1,CN19_Nisk_Dav_Dawn_100m_1,CN19_Nisk_Dav_Dawn_100m_2,CN19_Nisk_Dav_Dawn_100m_3,CN19_Nisk_Far_13m_1,CN19_Nisk_Far_13m_2,CN19_Nisk_Far_60m_1,CN19_Nisk_Far_60m_2,CN19_Nisk_Far_60m_3,CN19_Nisk_MO_Night_7m_2
589,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Dinophysiales,Dinophysiaceae,Dinophysis,unassigned,ASV_19569,0.0,...,0.0,0.0,0.0,0.0,4.0,2.0,0.0,0.0,0.0,0.0
590,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gonyaulacales,Gonyaulacaceae,Alexandrium,unassigned,ASVs_X,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
591,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gonyaulacales,Gonyaulacaceae,g_,s_,ASV_20582,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
592,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gonyaulacales,unassigned,unassigned,unassigned,ASVs_X,0.0,...,0.0,0.0,3.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0
593,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gymnodiniales,Gymnodiniaceae,Gymnodinium,Gymnodinium catenatum,ASVs_X,0.0,...,0.0,2.0,1.0,0.0,0.0,0.0,10.0,9.0,10.0,0.0
594,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gymnodiniales,Gymnodiniaceae,Gymnodinium,Gymnodinium sp. NVA/RUS/2008,ASVs_X,0.0,...,0.0,2.0,0.0,0.0,0.0,0.0,6.0,9.0,1.0,0.0
595,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gymnodiniales,Gymnodiniaceae,Gymnodinium,unassigned,ASV_45254,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
596,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gymnodiniales,Gymnodiniaceae,Lepidodinium,Lepidodinium chlorophorum,ASV_20640,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
597,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gymnodiniales,Gymnodiniaceae,Margalefidinium,Margalefidinium polykrikoides,ASVs_X,4.0,...,12.0,5.0,0.0,0.0,3.0,8.0,6.0,9.0,2.0,5.0
598,Eukaryota,unassigned,Dinophyceae,Dinophyceae,Gymnodiniales,Gymnodiniaceae,Margalefidinium,s_,ASV_14153,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Generate dataframes for (1) phylum- and class-level taxon frequences per sample and (2) reads and bps per sample

### COI

In [14]:
df_tax_COI = []
df_freq_COI = []
df_reads_COI = []

for file in glob.glob('*mgnmatches_tax.csv'):
    taxa_filt, taxa_freq, reads = counts_to_freq(file, 'COI', 321)
    counts = reformat_ASV_table(taxa_filt, 'COI')
    freq = reformat_ASV_table(taxa_freq, 'COI')
    df_tax_COI.append(counts)
    df_freq_COI.append(freq)
    df_reads_COI.append(reads)

df_tax_COI = pd.concat(df_tax_COI, axis=0)
df_freq_COI = pd.concat(df_freq_COI, axis=0)
df_reads_COI = pd.concat(df_reads_COI)

In [30]:
#df_tax_COI.head(3)
df_freq_COI.head(3)
#df_reads_COI.head(3)

,Domain,Phylum,Class,Order,Family,Genus,Species,COI sample,COI reads
5,Bacteria,Proteobacteria,Gammaproteobacteria,unknown,unknown,g_,s_,Lasker18Sc122_3_D0052,0.000235
11,Eukaryota,Ascomycota,Eurotiomycetes,Chaetothyriales,Trichomeriaceae,g_,s_,Lasker18Sc122_3_D0052,0.000088
13,Eukaryota,Ascomycota,Eurotiomycetes,Eurotiales,Aspergillaceae,g_,s_,Lasker18Sc122_3_D0052,0.000587


In [15]:
df_freq_COI['Domain'].unique()

array(['Bacteria', 'Eukaryota', 'unassigned'], dtype=object)

In [16]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/COI comparison/")
df_freq_COI.to_csv("COI_frequencies_all_tax_ranks_withBacUnass.csv", index=None)
df_tax_COI.to_csv("COI_readcounts_all_tax_ranks_withBacUnass.csv", index=None)
df_reads_COI.to_csv("COI_reads_bps_total.csv", index=None)

### 18S

In [17]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/18S comparison/ASV tables")

In [21]:
file = 'RREAS2019_18S_asv_filtered_noCs_mgnmatches_tax.csv'
gene = '18S'
len = 180

In [22]:
taxa_filt, taxa_freq, reads = counts_to_freq(file, gene, len)
#counts = reformat_ASV_table(taxa_filt, '18S')

In [25]:
taxa_freq[taxa_freq['Class'] == 'Prymnesiales']

,Domain,Kingdom,Phylum,Class,Order,Family,Genus,Species,ASV ID,CN19_Nisk_SM_Night_5m_1,...,CN19_Nisk_Dav_Dawn_5m_2,CN19_Nisk_Dav_Dawn_27m_1,CN19_Nisk_Dav_Dawn_100m_1,CN19_Nisk_Dav_Dawn_100m_2,CN19_Nisk_Dav_Dawn_100m_3,CN19_Nisk_Far_13m_1,CN19_Nisk_Far_13m_2,CN19_Nisk_Far_60m_1,CN19_Nisk_Far_60m_2,CN19_Nisk_Far_60m_3


In [18]:
df_tax_18S = []
df_freq_18S = []
df_reads_18S = []

for file in glob.glob('*mgnmatches_tax.csv'):
    taxa_filt, taxa_freq, reads = counts_to_freq(file, '18S', 180)
    counts = reformat_ASV_table(taxa_filt, '18S')
    freq = reformat_ASV_table(taxa_freq, '18S')
    df_tax_18S.append(counts)
    df_freq_18S.append(freq)
    df_reads_18S.append(reads)

df_tax_18S = pd.concat(df_tax_18S, axis=0)
df_freq_18S = pd.concat(df_freq_18S, axis=0) #.drop(columns="Unnamed: 0")
df_reads_18S = pd.concat(df_reads_18S)

In [19]:
#df_tax_18S.head(3)
df_freq_18S.head(3)
#df_reads_18S.head(3)

,Domain,Phylum,Class,Order,Family,Genus,Species,18S sample,18S reads
4,Eukaryota,Ascomycota,Eurotiomycetes,Eurotiales,Aspergillaceae,Penicillium,unassigned,CN18FESPkoa_SC24,0.00002
8,Eukaryota,Ascomycota,unassigned,unassigned,unassigned,unassigned,unassigned,CN18FESPkoa_SC24,0.00002
13,Eukaryota,Basidiomycota,Malasseziomycetes,Malasseziales,Malasseziaceae,Malassezia,Malassezia globosa,CN18FESPkoa_SC24,0.001325


In [20]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/18S comparison")
df_tax_18S.to_csv("18S_readcounts_all_tax_ranks_withBacUnass.csv", index=None)
df_freq_18S.to_csv("18S_frequencies_all_tax_ranks_withBacUnass.csv", index=None)
df_reads_18S.to_csv("18S_reads_bps_total.csv", index=None)

## 04. Compare total numbers/fractions of eukaryotic reads in metagenomes and amplicons

#### Calculate average # total reads and average # eukaryotic reads and bps for amplicon samples

In [3]:
def calculate_eukaryotic_read_fraction(df_tax, df_reads, marker_gene):
    domains = df_tax[['Domain', '%s sample' % marker_gene, '%s reads' % marker_gene]]
    domains = domains.groupby(['Domain', '%s sample' % marker_gene]).sum()
    euks = domains.xs('Eukaryota', level=0).reset_index()
    unass = domains.xs('unassigned', level=0).reset_index()
    euk_fracs = pd.merge(df_reads, euks, on='%s sample' % marker_gene).rename(columns={'%s reads' % marker_gene : '%s euk reads' % marker_gene})
    unass_fracs = pd.merge(df_reads, unass, on='%s sample' % marker_gene).rename(columns={'%s reads' % marker_gene : '%s unclass reads' % marker_gene})
    euk_fracs['%s euk reads percent' % marker_gene] = euk_fracs['%s euk reads' % marker_gene] / euk_fracs['total %s reads' % marker_gene]
    unass_fracs['%s unclass reads percent' % marker_gene] = unass_fracs['%s unclass reads' % marker_gene] / unass_fracs['total %s reads' % marker_gene]
    domain_fracs = euk_fracs.merge(unass_fracs, how='outer', on=['%s sample' % marker_gene, 'total %s bps' % marker_gene])
    if marker_gene == '18S' : 
        domain_fracs['%s prok reads percent' % marker_gene] = 1 - domain_fracs['%s euk reads percent' % marker_gene] - domain_fracs['%s unclass reads percent' % marker_gene]
    elif marker_gene == 'COI' : 
        domain_fracs['%s prok reads percent' % marker_gene] = 0
    domain_fracs['%s euk bps' % marker_gene] = domain_fracs['%s euk reads percent' % marker_gene] * domain_fracs['total %s bps' % marker_gene]
    domain_fracs['%s prok bps' % marker_gene] = domain_fracs['%s prok reads percent' % marker_gene] * domain_fracs['total %s bps' % marker_gene]
    domain_fracs['%s unclass bps' % marker_gene] = domain_fracs['%s unclass reads percent' % marker_gene] * domain_fracs['total %s bps' % marker_gene]
    domain_fracs = domain_fracs.drop('total %s reads_y' % marker_gene, axis=1)
    domain_fracs = domain_fracs.rename(columns={'total %s reads_x' % marker_gene : 'total %s reads' % marker_gene})
    return(domain_fracs)

#### Read in existing COI and 18S files

In [4]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/COI comparison/")
df_reads_COI = pd.read_csv("COI_reads_bps_total.csv")
df_tax_COI = pd.read_csv("COI_readcounts_all_tax_ranks_withBacUnass.csv")

In [5]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/18S comparison")
df_reads_18S = pd.read_csv("18S_reads_bps_total.csv")
df_tax_18S = pd.read_csv("18S_readcounts_all_tax_ranks_withBacUnass.csv")

In [6]:
df_reads_COI.head()

,COI sample,total COI reads,total COI bps
0,Lasker18Sc122_3_D0052,34063.0,10934223.0
1,Lasker18Sc122_5_D0053,64735.0,20779935.0
2,Lasker18Sc122_17_D0057,76319.0,24498399.0
3,Lasker18Sc122_19_D0058,69006.0,22150926.0
4,Lasker18Sc123_3_D0063,35687.0,11455527.0


In [7]:
domain_fracs_18S = calculate_eukaryotic_read_fraction(df_tax_18S, df_reads_18S, '18S')
domain_fracs_18S.head(3)

,18S sample,total 18S reads,total 18S bps,18S euk reads,18S euk reads percent,18S unclass reads,18S unclass reads percent,18S prok reads percent,18S euk bps,18S prok bps,18S unclass bps
0,CN18FESPkoa_SC24,99632,17933760,69396,0.696523,30236,0.303477,0.000000e+00,12491280.0,0.000000e+00,5442480.0
1,CN18FESPkoa_SC26,45794,8242920,29464,0.643403,16330,0.356597,-5.551115e-17,5303520.0,-4.575740e-10,2939400.0
2,CN18FESPkoa_SC29,90730,16331400,63989,0.705268,26741,0.294732,-5.551115e-17,11518020.0,-9.065748e-10,4813380.0


In [8]:
domain_fracs_COI = calculate_eukaryotic_read_fraction(df_tax_COI, df_reads_COI, 'COI')
domain_fracs_COI.head(3)

,COI sample,total COI reads,total COI bps,COI euk reads,COI euk reads percent,COI unclass reads,COI unclass reads percent,COI prok reads percent,COI euk bps,COI prok bps,COI unclass bps
0,CN18FESPkoa_SC24,81287.0,26093127.0,13975.0,0.171922,67312.0,0.828078,0,4485975.0,0.0,21607152.0
1,CN18FESPkoa_SC26,101830.0,32687430.0,19679.0,0.193253,82151.0,0.806747,0,6316959.0,0.0,26370471.0
2,CN18FESPkoa_SC29,88847.0,28519887.0,12536.0,0.141096,76311.0,0.858904,0,4024056.0,0.0,24495831.0


#### Import contig numbers and calculate averages across data sets

In [9]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/mmseqs")
contig_frac_df = pd.read_csv("mmseqs_contig_counts_by_domain_withunass.csv")

In [10]:
contig_frac_df.head()

,Metagenome sample,euk_contig_percent,prok_contig_percent,unass_contig_percent,Total_contigs
0,1903_buk5-1,0.185821,0.304237,0.509942,722829
1,1903_buk5-2,0.177641,0.316295,0.506064,721343
2,1903_buk6-1,0.057686,0.554939,0.387375,570377
3,1903_buk6-2,0.050693,0.579238,0.370069,352602
4,1903buk1_0m-1,0.129014,0.494236,0.376750,3246


In [11]:
avg_total_contigs = contig_frac_df['Total_contigs'].mean()
avg_euk_contigs = contig_frac_df['euk_contig_percent'].mean()
avg_prok_contigs = contig_frac_df['prok_contig_percent'].mean()
avg_unass_contigs = contig_frac_df['unass_contig_percent'].mean()

In [12]:
avg_euk_contigs

np.float64(0.06466607581716562)

#### Add average euk contigs and 18S and COI average euk reads to data frame with metagenome average euk bps

In [13]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/sourmash/summaries/dataset")
euk_frac_mgns = pd.read_csv("Metagenome_eukaryotic_reads_bps.csv")

In [14]:
euk_frac_mgns

,Sequence type,Avg num bps or contigs,Avg % euk bps or contigs,Avg % prok bps or contigs,Avg % unass bps or contigs,Avg num euk bps or contigs,Avg num prok bps or contigs,Avg num unass bps or contigs
0,Metagenome reads,3.092193e+10,0.019412,0.227567,0.753592,1.921727e+10,5.785672e+11,1.649386e+12


In [15]:
new_row = pd.DataFrame({'Sequence type': 'Metagenome contigs', 
                              'Avg num bps or contigs': [avg_total_contigs],
                           'Avg % euk bps or contigs' : [avg_euk_contigs], 
                              'Avg % prok bps or contigs' : [avg_prok_contigs],
                             'Avg % unass bps or contigs' : [avg_unass_contigs],
                        'Avg num euk bps or contigs': euk_frac_mgns['Avg % euk bps or contigs'] * euk_frac_mgns['Avg num bps or contigs'],
                        'Avg num prok bps or contigs': euk_frac_mgns['Avg % prok bps or contigs'] * euk_frac_mgns['Avg num bps or contigs'],
                        'Avg num unass bps or contigs': euk_frac_mgns['Avg % unass bps or contigs'] * euk_frac_mgns['Avg num bps or contigs']},
                        index=[0])
euk_frac_df = pd.concat([new_row, euk_frac_mgns.loc[:]]).reset_index(drop=True)

In [16]:
euk_frac_df

,Sequence type,Avg num bps or contigs,Avg % euk bps or contigs,Avg % prok bps or contigs,Avg % unass bps or contigs,Avg num euk bps or contigs,Avg num prok bps or contigs,Avg num unass bps or contigs
0,Metagenome contigs,1.459982e+06,0.064666,0.510098,0.425236,6.002555e+08,7.036821e+09,2.330252e+10
1,Metagenome reads,3.092193e+10,0.019412,0.227567,0.753592,1.921727e+10,5.785672e+11,1.649386e+12


In [17]:
new_row = pd.DataFrame({'Sequence type': '18S', 
           'Avg num bps or contigs': np.average(domain_fracs_18S['total 18S bps']),
                        'Avg % euk bps or contigs': np.average(domain_fracs_18S['18S euk reads percent']),
                        'Avg % prok bps or contigs': np.average(domain_fracs_18S['18S prok reads percent']),
                        'Avg % unass bps or contigs': np.average(domain_fracs_18S['18S unclass reads percent']),
                        'Avg num euk bps or contigs': np.average(domain_fracs_18S['18S euk bps']),
                        'Avg num prok bps or contigs': np.average(domain_fracs_18S['18S prok bps']),
                        'Avg num unass bps or contigs': np.average(domain_fracs_18S['18S unclass bps'])},
                      index=[0])
euk_frac_df = pd.concat([new_row, euk_frac_df.loc[:]]).reset_index(drop=True)

In [18]:
new_row = pd.DataFrame({'Sequence type': 'COI', 
           'Avg num bps or contigs': np.average(domain_fracs_COI['total COI bps']),
                        'Avg % euk bps or contigs': np.average(domain_fracs_COI['COI euk reads percent']),
                        'Avg % prok bps or contigs': np.average(domain_fracs_COI['COI prok reads percent']),
                        'Avg % unass bps or contigs': np.average(domain_fracs_COI['COI unclass reads percent']),
                        'Avg num euk bps or contigs': np.average(domain_fracs_COI['COI euk bps']),
                        'Avg num prok bps or contigs': np.average(domain_fracs_COI['COI prok bps']),
                        'Avg num unass bps or contigs': np.average(domain_fracs_COI['COI unclass bps'])},
                      index=[0])
euk_frac_df = pd.concat([new_row, euk_frac_df.loc[:]]).reset_index(drop=True)

In [19]:
euk_frac_df['Total percentage'] = 1.0
euk_frac_df

,Sequence type,Avg num bps or contigs,Avg % euk bps or contigs,Avg % prok bps or contigs,Avg % unass bps or contigs,Avg num euk bps or contigs,Avg num prok bps or contigs,Avg num unass bps or contigs,Total percentage
0,COI,2.987698e+07,0.662782,0.000000e+00,0.341778,2.024482e+07,0.000000e+00,9.751899e+06,1.0
1,18S,1.192126e+07,0.885346,3.101718e-18,0.114654,1.080101e+07,2.282518e-11,1.120245e+06,1.0
2,Metagenome contigs,1.459982e+06,0.064666,5.100978e-01,0.425236,6.002555e+08,7.036821e+09,2.330252e+10,1.0
3,Metagenome reads,3.092193e+10,0.019412,2.275673e-01,0.753592,1.921727e+10,5.785672e+11,1.649386e+12,1.0


## 05. Compare frequencies of taxa seen in paired samples: metagenome contigs vs amplicons

In [21]:
os.chdir("/Users/nastassiapatin/GitHub/Euk-eDNA-comparison/")

In [22]:
def combine_ASV_metagenome_freqs(sample_key, mgn_df, asv_df, marker_gene, tax_level1, tax_level2): 
    asv_df = asv_df.rename(columns={'%s sample' % marker_gene: 'ASV sample'})
    asv_df_taxon = pd.DataFrame(asv_df[['%s' % tax_level1,
                                        '%s' % tax_level2,
                                        'ASV sample', 
                                        '%s reads' % marker_gene]].groupby(['%s' % tax_level1,
                                                                            '%s' % tax_level2,
                                                                           'ASV sample'],
                                                                           as_index=False).sum())
    df_tax_asvs_paired = asv_df_taxon.merge(sample_key, how='left', on='ASV sample')
    df_tax_mgns_paired = mgn_df.merge(sample_key, how='left', on='Metagenome sample')
    return(df_tax_asvs_paired, df_tax_mgns_paired)

#### Load file showing paired sample names and turn pairs into a dictionary

In [23]:
file = 'eCruises2018_2019_mgn_asv_paired_samples.csv'
pairs = pd.read_csv(file)
pairs.head(3)

,ASV sample,Metagenome sample
0,CN18FESPkoa_SC24,zr2760_52
1,CN18FESPkoa_SC26,zr2760_64
2,CN18FESPkoa_SC29,zr2760_79


##### Import existing dfs

In [24]:
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/18S comparison")
df_freq_18S = pd.read_csv("18S_frequencies_all_tax_ranks_withBacUnass.csv")
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/COI comparison")
df_freq_COI = pd.read_csv("COI_frequencies_all_tax_ranks_withBacUnass.csv")
os.chdir("/Users/nastassiapatin/OneDrive - SCCWRP/Euks_mgns_asvs/mmseqs")
mmseqs_tax = pd.read_csv("contigs_mmseqs_Phylum_Class.csv")

In [7]:
mmseqs_tax.head()

,Metagenome sample,p,c,percent_TPM
0,1903_buk5-1,Annelida,Clitellata,0.000008
1,1903_buk5-1,Annelida,Polychaeta,0.000028
2,1903_buk5-1,Annelida,Unclassified,0.000003
3,1903_buk5-1,Apicomplexa,Aconoidasida,0.000011
4,1903_buk5-1,Apicomplexa,Conoidasida,0.000031


In [25]:
mmseqs_tax.loc[:, 'p'] = np.where(mmseqs_tax['p'] == 'Unclassified', mmseqs_tax['c'], mmseqs_tax['p'])

In [26]:
mmseqs_tax = mmseqs_tax.rename(columns = {'p' : 'Phylum', 'c' : 'Class'})
mmseqs_tax.head()

,Metagenome sample,Phylum,Class,percent_TPM
0,1903_buk5-1,Annelida,Clitellata,0.000008
1,1903_buk5-1,Annelida,Polychaeta,0.000028
2,1903_buk5-1,Annelida,Unclassified,0.000003
3,1903_buk5-1,Apicomplexa,Aconoidasida,0.000011
4,1903_buk5-1,Apicomplexa,Conoidasida,0.000031


### Change "unassigned" annotations to "Unclassified" for amplicons

In [28]:
df_freq_18S.loc[df_freq_18S['Phylum'] == 'unassigned', 'Phylum'] = "Unclassified"
df_freq_COI.loc[df_freq_COI['Phylum'] == 'unassigned', 'Phylum'] = "Unclassified"
df_freq_18S.loc[df_freq_18S['Class'] == 'unassigned', 'Class'] = "Unclassified"
df_freq_COI.loc[df_freq_COI['Class'] == 'unassigned', 'Class'] = "Unclassified"
df_freq_18S.loc[df_freq_18S['Phylum'] == 'unknown', 'Phylum'] = "Unclassified"
df_freq_COI.loc[df_freq_COI['Phylum'] == 'unknown', 'Phylum'] = "Unclassified"
df_freq_18S.loc[df_freq_18S['Class'] == 'unknown', 'Class'] = "Unclassified"
df_freq_COI.loc[df_freq_COI['Class'] == 'unknown', 'Class'] = "Unclassified"

### Parse metagenomes and each marker gene separately

In [29]:
# 18S
asvs, mgns = combine_ASV_metagenome_freqs(pairs, 
                                          mmseqs_tax, 
                                          df_freq_18S, 
                                          '18S', 
                                          'Phylum',
                                         'Class')

In [36]:
# COI
asvs, mgns = combine_ASV_metagenome_freqs(pairs, 
                                          mmseqs_tax, 
                                          df_freq_COI, 
                                          'COI', 
                                          'Phylum', 
                                          'Class')

### Combine ASV and metagenome taxa frequences at the Phylum AND Class levels

In [37]:
df_tax_pairmatches = mgns.merge(asvs, how='outer', 
                                            on=['Phylum', 
                                                'Class',
                                                'ASV sample', 
                                                'Metagenome sample'])
df_tax_pairmatches.head(10)

,Metagenome sample,Phylum,Class,percent_TPM,ASV sample,COI reads
0,zr2760_52,Acantharea,Acantharea,2.766264e-07,CN18FESPkoa_SC24,NaN
1,zr2760_64,Acantharea,Acantharea,3.970424e-07,CN18FESPkoa_SC26,NaN
2,zr2760_79,Acantharea,Acantharea,1.423334e-06,CN18FESPkoa_SC29,NaN
3,zr2760_69,Acantharea,Acantharea,4.140965e-07,CN18Fc47_6_eDNA,NaN
4,D0052A,Acantharea,Acantharea,3.073382e-07,Lasker18Sc122_3_D0052,NaN
5,D0053A,Acantharea,Acantharea,1.477337e-07,Lasker18Sc122_5_D0053,NaN
6,D0128A,Acantharea,Acantharea,1.206552e-07,Lasker18Sc130_5_D0128,NaN
7,D0174A,Acantharea,Acantharea,1.174835e-07,Lasker18Sc135_19_D0174,NaN
8,zr2760_47,Acanthocephala,Palaeacanthocephala,5.933312e-07,CN18Fc36_6_eDNA,NaN
9,zr2760_29,Acanthocephala,Palaeacanthocephala,1.926149e-07,CN18Sc18_2_eDNA,NaN


## In cases where a sample has no matching sample containing any of the same phylum/class combo, need to add in the corresponding sample name and replace content with '0'

In [38]:
# Create a dictionary 
value_map = pairs.set_index('ASV sample')['Metagenome sample'].to_dict()
# Use map to replace NaN 'Metagenome sample' values in df_tax_pairmatches
df_tax_pairmatches['Metagenome sample'] = df_tax_pairmatches.apply(lambda row: 
                                                            value_map[row['ASV sample']] if pd.isna(row['Metagenome sample']) 
                                                            else row['Metagenome sample'], axis=1)
# Create the inverse dictionary
value_map = pairs.set_index('Metagenome sample')['ASV sample'].to_dict()
# Use map to replace NaN 'ASV sample' values in df_tax_pairmatches
df_tax_pairmatches['ASV sample'] = df_tax_pairmatches.apply(lambda row: 
                                                            value_map[row['Metagenome sample']] if pd.isna(row['ASV sample']) 
                                                            else row['ASV sample'], axis=1)

In [39]:
df_tax_pairmatches

,Metagenome sample,Phylum,Class,percent_TPM,ASV sample,COI reads
0,zr2760_52,Acantharea,Acantharea,2.766264e-07,CN18FESPkoa_SC24,NaN
1,zr2760_64,Acantharea,Acantharea,3.970424e-07,CN18FESPkoa_SC26,NaN
2,zr2760_79,Acantharea,Acantharea,1.423334e-06,CN18FESPkoa_SC29,NaN
3,zr2760_69,Acantharea,Acantharea,4.140965e-07,CN18Fc47_6_eDNA,NaN
4,D0052A,Acantharea,Acantharea,3.073382e-07,Lasker18Sc122_3_D0052,NaN
...,...,...,...,...,...,...
41461,D0174A,Zoopagomycota,Zoopagomycetes,2.761803e-06,Lasker18Sc135_19_D0174,NaN
41462,D0275A,Zoopagomycota,Zoopagomycetes,1.530558e-06,Lasker18Sc158_17_D0275,NaN
41463,D0276A,Zoopagomycota,Zoopagomycetes,1.399810e-06,Lasker18Sc158_19_D0276,NaN
41464,D0270A,Zoopagomycota,Zoopagomycetes,1.000132e-06,Lasker18Sc158_3_D0270,NaN


In [40]:
# Replace NaN in 'percent_TPM' and 'COI reads' or '18S reads' columns with '0'
df_tax_pairmatches['percent_TPM'] = df_tax_pairmatches['percent_TPM'].fillna(0)
#df_tax_pairmatches['18S reads'] = df_tax_pairmatches['18S reads'].fillna(0)
df_tax_pairmatches['COI reads'] = df_tax_pairmatches['COI reads'].fillna(0)

#### Need this step to avoid 'nan' values in the exported file!

In [41]:
df_tax_pairmatches['Class']= df_tax_pairmatches['Class'].astype(pd.StringDtype())
df_tax_pairmatches.dtypes

Metagenome sample            object
Phylum                       object
Class                string[python]
percent_TPM                 float64
ASV sample                   object
COI reads                   float64
dtype: object

#### COI

In [42]:
os.chdir("/Users/nastassiapatin/GitHub/Euk-eDNA-comparison/COI comparison")
df_tax_pairmatches.to_csv("Phylum_Class_COI_samplepairs.csv", index=False)

#### 18S

In [ ]:
os.chdir("/Users/nastassiapatin/GitHub/Euk-eDNA-comparison/18S comparison")
df_tax_pairmatches.to_csv("Phylum_Class_18S_samplepairs.csv", index=False)